In [2]:
import random

# 不同约束的难度可能会影响实验结果
我们还需要测试单条指令的遵循率
因此可以设计在固定深度d=x的情况下，单独测试每条指令遵循的成功率，以及一些指令组合的遵循成功率。

In [8]:
AMAX = 12 # 单步运算第二个操作数的上限

CONSTRAINS = [
'''
The final line of your response MUST be exactly in this format:
####ANSWER=<answer>
Where <answer> is your answer
Do not write anything after the #### line.
''',
'''The answer MUST be rounded to one decimal place following the rule of rouding off, including cases where the decimal is zero. 
''',
'''The numbers after the decimal point MUST be expressed using English words.
''',
'''The english words in the answer MUST in upper case.
''',
'''While following other rules, output the integer part of the answer in reverse order. For example: 123 -> 321
''',
'''The final answer MUST begin with the exact prefix "RESULT:".
''',
'''The final answer MUST contain no more than 12 whitespace-separated tokens, including the required answer marker.
''',
'''Replace every occurrence of the digit 0 with Z, every occurrence of the digit 1 with O, and every occurrence of the digit 2 with T. Leave all other characters unchanged.
''',
'''Append the sum of all decimal digits in the rounded numerical answer as a suffix in square brackets. Do not include the decimal point or the negative sign in the sum.
''',
'''The first line of response MUST be 'HELLO WORLD'.
'''
]

def _make_constrain(k):
    ''' 约束最少为1'''
    if k > len(CONSTRAINS):
        k = len(CONSTRAINS)
    constrain = ''
    for i in range(k):
        constrain = constrain + CONSTRAINS[i]

    # print(constrain)
    return constrain

def _apply(y, kind, a):
    if kind == '+': return (y + a)
    if kind == '-': return (y - a)
    if kind == '/': return (y / a)
    return (y * a)          

def _draw(rng, n, amax):
    return [(rng.choice('+-*/'), rng.randrange(2, amax + 1)) for _ in range(n)]

def make_question_with_d_and_k(rng, d, k, amax=AMAX):
    '''串行链：深度 d，约束k'''
    upper_limit = 100
    x = rng.randrange(2, upper_limit)
    ops = _draw(rng, d, amax)
    y = x
    constrain = _make_constrain(k)
    for symbol, a in ops:
        y = _apply(y, symbol, a)
    body = '\n'.join(f'{i+1}. ( {k} {a} )' for i, (k, a) in enumerate(ops))
    q = (f'Start with x = {x}. Apply the following operations IN ORDER.'
         f'\n'
         f'{body}\nWhat is the final value of x?')
    return dict(q=q, ans=y, d=d, W=d, cons = constrain)

q = make_question_with_d_and_k(random.Random(42), 10, 5)

print(q['q'])
print(q['ans'])
print(q['cons'])

Start with x = 83. Apply the following operations IN ORDER.
1. ( + 2 )
2. ( * 5 )
3. ( - 4 )
4. ( + 12 )
5. ( + 11 )
6. ( / 2 )
7. ( + 3 )
8. ( - 5 )
9. ( + 10 )
10. ( - 12 )
What is the final value of x?
218.0

The final line of your response MUST be exactly in this format:
####ANSWER=<answer>
Where <answer> is your answer
Do not write anything after the #### line.
The answer MUST be rounded to one decimal place following the rule of rouding off, including cases where the decimal is zero. 
The numbers after the decimal point MUST be expressed using English words.
The english words in the answer MUST in upper case.
While following other rules, output the integer part of the answer in reverse order. For example: 123 -> 321



# Generate the whole dataset

In [11]:
random_seed_generator = random.Random(42)
max_d = 20
max_k = 10
q_in_each_grid = 20
q_matrix = []
for d in range(1, max_d + 1):
    k_list = []
    for k in range(1, max_k + 1):
        q_list = []
        for i in range(q_in_each_grid):
            random_seed = random_seed_generator.randint(0, 2**32)
            q = make_question_with_d_and_k(random.Random(random_seed), d, k)

            # 超级拼装这一块
            q_list.append(q)
        k_list.append(q_list)
    q_matrix.append(k_list)

# q[depth][constrain_num][question_number]
print(q_matrix[1][9][1])

{'q': 'Start with x = 45. Apply the following operations IN ORDER.\n1. ( - 2 )\n2. ( * 4 )\nWhat is the final value of x?', 'ans': 172, 'd': 2, 'W': 2, 'cons': '\nThe final line of your response MUST be exactly in this format:\n####ANSWER=<answer>\nWhere <answer> is your answer\nDo not write anything after the #### line.\nThe answer MUST be rounded to one decimal place following the rule of rouding off, including cases where the decimal is zero. \nThe numbers after the decimal point MUST be expressed using English words.\nThe english words in the answer MUST in upper case.\nWhile following other rules, output the integer part of the answer in reverse order. For example: 123 -> 321\nThe final answer MUST begin with the exact prefix "RESULT:".\nThe final answer MUST contain no more than 12 whitespace-separated tokens, including the required answer marker.\nReplace every occurrence of the digit 0 with Z, every occurrence of the digit 1 with O, and every occurrence of the digit 2 with T. L

In [12]:
import json
from datasets import load_dataset, Dataset

rows = []
for d, k_list in enumerate(q_matrix):
    for k, q_list in enumerate(k_list):
        for i, q in enumerate(q_list):
            rows.append({
                'd': d,
                'k': k,
                'i': i,
                'q': q['q'],
                'ans': q['ans'],
                'W': q['W'],
                'cons': q['cons']
            })

save_path = 'synthesis_q_k_dataset.jsonl'

with open(save_path, 'w', encoding='utf-8') as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii = False) + '\n')

ds = load_dataset('json', data_files=save_path , split='train')
print(ds)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['d', 'k', 'i', 'q', 'ans', 'W', 'cons'],
    num_rows: 4000
})
